## Observações

- RFE não apresentou ganho de desempenho em relação ao SelectKBest e aumentou significativamente o custo computacional.
- A imputação pela média apresentou melhor desempenho empírico que a imputação pela moda.
- `mutual_info_classif` apresentou melhor desempenho entre as funções de seleção de avaliadas, incluindo `f_classif` e `chi2`.
- `discrete_features=True` apresentou melhor desempenho que `discrete_features=False` em `mutual_info_classif`, mesmo quando imputando pela média.
- Oversampling melhorou levemente as métricas de´desempenho, especialmente o `recall`.
- O melhor desempenho observado foi obtido com aproximadamente cinco loci. O uso de 50 loci aumentou o sobreajuste, provavelmente devido à alta dimensionalidade, redundância, pequeno número de amostras e/ou fatores de confusão.
- A estabilidade dos loci selecionados entre diferentes partições ainda precisa ser avaliada antes de interpretá-los biologicamente.
- O XGBoost apresentou melhor desempenho, com `balanced_accuracy` de aproximadamente 0,64.
- O `recall` da classe positiva permaneceu próximo de 0,5, indicando que o limiar padrão não produz a relação desejada entre falsos positivos e falsos negativos.

## Decisões

- Remover `RFE` do pipeline e manter apenas `SelectKBest` com `mutual_info_classif(discrete_features=True)`.
- Manter por volta de 5 genes na seleção de features, com possibilidade de ajuste fino.
- Manter `SimpleImputer(strategy="mean")` para imputação de valores ausentes.
- Tentar técnicas para melhorar o `recall`, e.g., ajuste de limiar de decisão, outros métodos de oversampling.

In [ ]:
from functools import partial

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from covid import feature
from covid.feature.column_dropper import ColumnDropper
from covid.feature.high_missing_rate_dropper import HighMissingRateDropper

mutual_information = partial(
    mutual_info_classif, discrete_features=True, random_state=42
)

classifier = LogisticRegression(
    solver="liblinear", C=0.01, max_iter=5000, random_state=42
)

pipeline = Pipeline(
    [
        ("dropper", ColumnDropper(columns_to_drop=feature.ID)),
        ("missing_rate_dropper", HighMissingRateDropper(missing_threshold=0.05)),
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=mutual_information, k=5)),
        ("sampler", RandomOverSampler(random_state=42)),
        ("classifier", classifier),
    ]
)
pipeline.set_output(transform="pandas")

In [ ]:
import pandas as pd
from covid import constants

train_data = pd.read_csv(constants.INTERIM_TRAIN_DATA_PATH, dtype={feature.ID: str})
train_data.shape

In [ ]:
X_train = train_data.drop(columns=[feature.TARGET])
y_train = train_data[feature.TARGET]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

param_grid = [
    {
        "classifier": [
            LogisticRegression(
                solver="liblinear", C=0.01, max_iter=5000, random_state=42
            )
        ],
        "classifier__solver": ["liblinear", "lbfgs"],
    },
    {
        "classifier": [SVC(random_state=42)],
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__kernel": ["linear", "rbf"],
        "classifier__gamma": ["scale", "auto"],
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [500, 800],
        "classifier__min_samples_leaf": [1, 3],
    },
    {
        "classifier": [
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
            )
        ],
        "classifier__n_estimators": [200, 500],
        "classifier__learning_rate": [0.03, 0.1],
        "classifier__max_depth": [2, 3],
        "classifier__subsample": [0.8, 0.5],
        "classifier__colsample_bytree": [0.7, 0.5],
        "classifier__reg_alpha": [0.1, 0.001],
        "classifier__reg_lambda": [3.0, 5.0],
    }
]

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV

metrics = ["balanced_accuracy", "recall", "f1", "precision", "roc_auc"]

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring=metrics,
    refit=metrics[0],
    cv=cv,
    return_train_score=True,
    verbose=2,
    n_jobs=-1,
)
search.fit(X_train, y_train)

In [ ]:
search.best_params_

In [ ]:
from covid.training.utils import summarize_grid_search_results

cv_summary = summarize_grid_search_results(search.cv_results_)
cv_summary.to_csv("summary.csv", index=False)
cv_summary.head(15)

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay

## This is a development evaluation only.
# These metrics should be interpreted carefully as they can be optimistic
scv = StratifiedKFold(shuffle=True, n_splits=5, random_state=42)
y_pred = cross_val_predict(
    search.best_estimator_, X_train, y_train, cv=scv, n_jobs=-1
)

ConfusionMatrixDisplay.from_predictions(y_train, y_pred)